In [ ]:
'''
liste d'extensions considérées comme "non-modélisation" (images, documents, notes)
qui seront redirigées vers le dossier metadata
'''

import shutil
import json
import time
import requests
import csv
from pathlib import Path
from bioservices import BioModels
import biomodels

# Root directory configuration
racine_data = Path("./BioModels_Data_cleaned_redirection_metadata_json_dico_") 
racine_data.mkdir(parents=True, exist_ok=True)

s = BioModels()

diseases = {
    "dengue_files": ('dengue','DENV'),
    "chikungunya_files": ("chikungunya", "CHIKV"),
    "lyme_files": ('lyme', 'borrelia', 'borreliosis'),
    "mpox_files": ('mpox', 'monkeypox'),
    "west_nile_files": ('west nile', 'WNV'),
    "influenza_files": ("influenza", "influenza virus","avian influenza", "H5N1"),
    "tuberculosis_files": ("tuberculosis", "TB", "mycobacterium"),
    "hiv_files": ("HIV", "human immunodeficiency virus"),
    "covid_files": ("covid", "SARS-CoV-2")
}

# Define extensions that should be moved to 'metadata' instead of 'model'
EXTENSIONS_METADATA = {'.png', '.jpg', '.jpeg', '.pdf', '.txt', '.docx', '.doc', '.xlsx', '.xls', '.csv'}

# List to store data for the final CSV report
csv_data = []

for folder_name, queries in diseases.items():
    print(f"\n--- Category: {folder_name} ---")
    query_string = " OR ".join(f'"{q}"' for q in queries)
    results = s.search(query_string, numResults=100) 
    
    if not isinstance(results, dict):
        print(f"Server error for {folder_name}. Waiting...")
        time.sleep(5)
        continue

    ids = [res['id'] for res in results.get('models', [])]
    time.sleep(1)

    for model_id in ids:
        try:
            # Create local directory structure
            dossier_base = racine_data / folder_name / model_id
            d_metadata = dossier_base / "metadata"
            d_model = dossier_base / "model"
            
            d_metadata.mkdir(parents=True, exist_ok=True)
            d_model.mkdir(parents=True, exist_ok=True)

            # Whitelist of files to keep in the metadata folder (prevent deletion during cleanup)
            files_to_preserve = [f"{model_id}_web_metadata.json", "files_list.json"]

            # 3. Download Web JSON Metadata
            print(f"  > Fetching Web JSON for {model_id}...")
            web_url = f"https://www.ebi.ac.uk/biomodels/{model_id}?format=json"
            try:
                web_res = requests.get(web_url, timeout=10)
                if web_res.status_code == 200:
                    with open(d_metadata / files_to_preserve[0], "w", encoding="utf-8") as f_web:
                        json.dump(web_res.json(), f_web, indent=4)
            except Exception:
                print(f"    ! Web JSON fetch failed")

            # 4. Retrieve internal files list from BioModels
            files_objects = biomodels.get_metadata(model_id)
            if not files_objects: continue
            
            with open(d_metadata / files_to_preserve[1], "w", encoding="utf-8") as f_meta:
                json.dump(files_objects, f_meta, indent=4, default=str)

            # Dictionary to count extensions for the current model
            stats_extensions = {}

            # 5. Download and sort files based on extension and name
            print(f"  > Downloading and sorting files for {model_id}...")
            for target_file_obj in files_objects:
                try:
                    nom_reel = getattr(target_file_obj, 'name', str(target_file_obj))
                    if not nom_reel or nom_reel == "None": continue

                    # Identify file extension
                    ext = Path(nom_reel).suffix.lower() or "no_extension"
                    stats_extensions[ext] = stats_extensions.get(ext, 0) + 1

                    # Sorting logic:
                    # Move to metadata if: specific keywords in name OR extension in EXTENSIONS_METADATA
                    is_metadata_keyword = any(x in nom_reel.lower() for x in ["metadata", ".json", ".rdf", ".owl"])
                    is_non_modeling_ext = ext in EXTENSIONS_METADATA
                    
                    if is_metadata_keyword or is_non_modeling_ext:
                        dest_path = d_metadata / nom_reel
                        files_to_preserve.append(nom_reel) # Add to whitelist to avoid deletion
                    else:
                        dest_path = d_model / nom_reel

                    # Download the file
                    result = biomodels.get_file(target_file_obj)
                    if isinstance(result, (str, Path)) and Path(result).exists():
                        shutil.copy(result, dest_path)
                    else:
                        with open(dest_path, "w", encoding="utf-8") as f:
                            f.write(str(result))
                except Exception:
                    print(f"    ! Error processing file: {nom_reel}")

            # 6. Final Cleanup: Remove any unexpected files from the metadata folder
            for file_path in d_metadata.iterdir():
                if file_path.name not in files_to_preserve:
                    file_path.unlink()

            # Prepare statistics row for the final CSV
            row = {"category": folder_name, "model_id": model_id}
            row.update(stats_extensions)
            csv_data.append(row)

            time.sleep(0.8) # Anti-throttling pause

        except Exception as e:
            print(f"Critical error on model {model_id}: {e}")

# 7. Generate the global CSV summary
if csv_data:
    # Identify all unique extensions found across all models for CSV headers
    all_cols = sorted(list(set().union(*(d.keys() for d in csv_data if isinstance(d, dict)))))
    with open(racine_data / "summary_extensions.csv", "w", newline="", encoding="utf-8") as f_csv:
        writer = csv.DictWriter(f_csv, fieldnames=all_cols)
        writer.writeheader()
        writer.writerows(csv_data)
    print(f"\nCSV Summary generated successfully at: {racine_data}/summary_extensions.csv")